# Sieve of Eratosthenes

In the `Primes.jl` demo, `primes(n)` handed us all primes up to $n$ in one line.

This notebook is about what's actually happening under the hood and why I think it's the most elegant algorithm in all of mathematics.

The problem is quite straightforward: find all prime numbers up to some limit $n$.

## The Naive Approach

The obvious approach is to test each number individually. You might call it "brute forcing".

For every number $k$ up to $n$, check if anything divides it evenly.

Like most algorithms in cryptography, it works for small $n$, but for large $n$ it's infeasible.

For each number you're computing up to $\sqrt{k}$ division checks, which is a time complexity of roughly $O(n\sqrt{n})$.

There's a much better way.

## The Sieve

The key insight is to flip the question. Instead of asking *"is this prime?"* for each number, assume everything is prime and cross out the composites.

The algorithm is:

1. Create a list of booleans, all set to `true`, indexed from $1$ to $n$.
2. Mark index $1$ as `false`, since $1$ is neither prime nor composite.
3. For each number $i$ from $2$ to $\sqrt{n}$: if $i$ is still marked `true`, cross out all multiples of $i$ starting from $i^2$.
4. Everything still marked `true` is prime.

Two things worth noting before I show you the code behind it:

- We only need to sieve up to $\sqrt{n}$ because any composite number $k$ must have a factor $\leq \sqrt{k}$.
- We start crossing out at $i^2$ rather than $2i$ because all smaller multiples of $i$ have already been crossed out by earlier primes.

In [8]:
function sieve(n)
    is_prime = fill(true, n + 1)
    is_prime[1] = false

    for i in 2:n
        if is_prime[i] && i*i <= n
            for j in i*i:i:n
                is_prime[j] = false
            end
        end
    end
    return [i for i in 1:n if is_prime[i]]
end

sieve (generic function with 1 method)

Let's verify this against `Primes.jl`:

In [9]:
using Primes
sieve(30) == primes(30)

true

In [ ]:
using CairoMakie
CairoMakie.activate!(type = "svg")

# Run sieve on 1-100
is_prime = fill(true, 101)
is_prime[1] = false

for i in 2:10
    if is_prime[i]
        for j in i*i:i:100
            is_prime[j] = false
        end
    end
end

# Visualization
fig = Figure(size=(700, 700))
ax = Axis(fig[1, 1], aspect=DataAspect(), title="Sieve of Eratosthenes (1-100)")
hidedecorations!(ax)

for row in 1:10
    for col in 1:10
        num = (row - 1) * 10 + col
        color = if num == 1
            RGBf(1, 1, 1)  # white for 1
        elseif is_prime[num]
            RGBf(0.85, 0.70, 0.15)  # gold for primes
        else
            RGBf(0.75, 0.75, 0.75)  # gray for composites
        end
        
        rect = Rect2f((col-1, 10-row), (1, 1))
        poly!(ax, rect, color=color, strokecolor=:black, strokewidth=1.5)
        text!(ax, col - 0.5, 10 - row + 0.5, text="$num", align=(:center, :center), fontsize=9)
    end
end

xlims!(ax, 0, 10)
ylims!(ax, 0, 10)
fig

## Visualization: The Sieve in Action

Let's visualize the sieve algorithm on a grid to see which numbers get crossed out:

## Why start at $i^2$?

When we reach $i$ in the outer loop, every multiple of $i$ is below $i^2$ and can be written as $i \times k$ where $k \lt i$.

Since $k \lt i$, we already processed $k$ in a previous iteration.

Starting at $i^2$ skips all that redundant work.

This is what gives the Sieve its time complexity $O(n\log \log n)$, which is nearly linear.

In [ ]:
using CairoMakie
import Statistics

# Compute prime counts using sieve
limit = 10000
prime_counts = []
ns = []

for n in [100, 500, 1000, 2000, 5000, 10000]
    primes = sieve(n)
    push!(ns, n)
    push!(prime_counts, length(primes))
end

# Prime number theorem approximation: π(n) ≈ n / ln(n)
pnt_approx = [n / log(n) for n in ns]

# Plot
fig = Figure(size=(700, 500))
ax = Axis(fig[1, 1], xlabel="n", ylabel="Prime Count π(n)", title="Prime Distribution vs Prime Number Theorem")

lines!(ax, ns, prime_counts, label="Actual π(n)", linewidth=3, color=:darkblue)
lines!(ax, ns, pnt_approx, label="Approximation n/ln(n)", linewidth=2, linestyle=:dash, color=:red)

axislegend(ax, position=:topleft)
fig

## Visualization: Prime Distribution

Here's how the actual prime count π(n) compares to the prime number theorem's approximation:

## Project Euler Problem 7

What is the $10001 \text{st}$ prime?

In [10]:
sieve(ceil(Int, 10001 * log(10001) * 1.2))[10001]

104743

## Project Euler Problem 10

Sum of all primes below $2$ million:

In [11]:
sum(sieve(2_000_000))

142913828922

## Limitations

Although the Sieve is fast, it's quite memory hungry. It requires a boolean array of size $n$. For $n = 600{,}851{,}475{,}143$ (Project Euler problem $3$ ), that's hundreds of gigabytes. In practice, segmented sieves or probabilistic primality tests like Miller-Rabin are used instead.

For cryptographic purposes, `nextprime` from `Primes.jl` is the practical tool since it uses Miller-Rabin internally and works at any scale.

The sieve's real value isn't as a production tool — it's as one of the clearest examples of algorithmic thinking in mathematics. A 2,200-year-old algorithm that still runs in nearly linear time.